# Fuji Film Recipe Recommender

Given a description of the current day (weather, light, mood, scene), suggest the best Fujifilm film simulation recipe to use.

In [22]:
# imports

import os
import json
import random
from dotenv import load_dotenv
from scraper import fetch_website_links, fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

MODEL = "llama3.2" #select the model you want here
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [23]:
# Fuji X Weekly has a dedicated recipes page per sensor generation - use it when the description names one,
# since it has far more (and more relevant) recipes than the general hub page.

SENSOR_RECIPE_PAGES = [
    ("xtrans v", "https://fujixweekly.com/fujifilm-x-trans-v-recipes/"),
    ("xtrans iv", "https://fujixweekly.com/fujifilm-x-trans-iv-recipes/"),
    ("xtrans iii", "https://fujixweekly.com/fujifilm-x-trans-iii-recipes/"),
    ("xtrans ii", "https://fujixweekly.com/fujifilm-x-trans-ii-recipes/"),
    ("xtrans i", "https://fujixweekly.com/fujifilm-x-trans-i-recipes/"),
    ("bayer", "https://fujixweekly.com/fujifilm-bayer-recipes/"),
    ("exrcmos", "https://fujixweekly.com/fujifilm-exr-cmos-film-simulation-recipes/"),
    ("gfx", "https://fujixweekly.com/fujifilm-gfx-recipes/"),
    ("full spectrum", "https://fujixweekly.com/full-spectrum-recipes/"),
]
RECIPES_HUB_URL = "https://fujixweekly.com/recipes/"

def get_recipe_links(description):
    normalized = description.lower().replace("-", "")
    url = RECIPES_HUB_URL
    for keyword, sensor_url in SENSOR_RECIPE_PAGES:
        if keyword in normalized:
            url = sensor_url
            break

    all_links = fetch_website_links(url)
    return sorted(set(
        link for link in all_links
        if link and "fujixweekly.com" in link and "recipe" in link.lower() and "comment" not in link.lower()
    ))

In [24]:
link_system_prompt = "You are given a description of a day/scene from a photographer, and a list of links to Fujifilm " \
"film simulation recipe pages. Pick the 1 to 3 links whose title looks most likely to match the description " \
"(consider mood, lighting, colors and, if mentioned, the camera model). " \
"Only pick from the links provided - never invent a URL. " \
"Respond in JSON as in this example:\n\n" \
'{"links": [{"name": "short title guessed from the url", "url": "https://full.url/goes/here"}]}'

In [25]:
def select_relevant_recipes(description):
    recipe_links = get_recipe_links(description)

    if "random" in description.lower():
        url = random.choice(recipe_links)
        return {"links": [{"name": "random recipe", "url": url}]}

    user_prompt = f"Description: {description}\n\nHere are the candidate recipe links:\n\n" + "\n".join(recipe_links)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [26]:
recommend_system_prompt = "You are a professional photographer for Fujifilm cameras. " \
"You will be given the contents of one or more real film simulation recipe pages from https://fujixweekly.com/. " \
"Recommend the recipe that best matches the user's description, including its camera/sensor compatibility and settings. " \
"Always cite the exact URL the recipe came from. " \
"Only use recipes that are actually present in the provided content - never invent one, and say so if none of them fit."

In [27]:
def fetch_candidate_recipe_content(description):
    candidates = select_relevant_recipes(description)
    content = ""
    for link in candidates["links"]:
        content += f"\n\n## Recipe page: {link['url']}\n"
        content += fetch_website_contents(link["url"])
    return content

In [28]:
def recommend_recipe(description):
    user_prompt = f"{description}\n\nHere are the candidate recipe pages to choose from:\n\n" + fetch_candidate_recipe_content(description)
    messages = [
        {"role": "system", "content": recommend_system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    display(Markdown(response.choices[0].message.content))

In [31]:
recommend_recipe(
    "Give me a random recipe. "
    "I use the camera Fujifilm XM5, so it should match the correct sensor that is X-TRANS IV. "
    "I also want the link for this random recipe."
)

Based on your request, I recommend the recipe published in:

**Recipe page:** https://fujixweekly.com/2021/07/24/fujifilm-x-trans-iv-film-simulation-recipe-fujicolor-super-hg-part-1-of-3/

This recipe matches the camera/sensor you specified, the Fujifilm X-TRANS IV. The recipe is for the Fujicolor Super HG film simulation, and it was created by Thomas Schwab.

The recommended settings are not explicitly stated in the text you provided, but you can try using a combination of the following settings to get a similar look:

* Film Simulation: Fujicolor Super HG
* White Balance: Fluorescent 2 (also called "Warm White Fluorescent" or "Neon 2")
* Shadows: -1
* Highlights: +1
* Color Grading: Saturation: +20%, Color: -10%

Keep in mind that these settings are not explicitly recommended in the original recipe page, and you may need to adjust them to get the look you want.